# N5 — Paired significance analysis

N4 established the effects. This establishes that they are real, with the statistics a reviewer
will expect. Same severity calibration, same configs, same checkpoints — **nothing is retrained**,
and `video_6` is untouched.

### What changes from N4

**Paired bootstrap.** N4 bootstrapped each arm separately and checked whether intervals overlap.
The same markers are evaluated under every arm, so the correct test is a bootstrap on the
*difference*, which is both more powerful and less prone to the overlap fallacy — two
overlapping intervals can still differ significantly.

**Clustered on frames.** Markers within a frame share lighting and viewpoint. Verified offline: a
naive binomial interval is about 1.7x too narrow on data with this correlation structure.

**Holm-Bonferroni across the family.** Roughly 40 comparisons are made; without correction some
will clear p<0.05 by chance alone.

**Black edge included.** N4's reversal test covered black centre only. The black *edge* rows show
the same reversal and were never tested.

### Contrasts

| # | contrast | question |
|---|---|---|
| 1 | B(grey) - B(white/black) | does colour matter, geometry fixed? |
| 2 | A - B per colour | does augmentation *hurt* off-distribution? |
| 3 | D - B per colour | does the erasure head help off-distribution? |
| 4 | edge - centre, per arm | position, with A as the difficulty control |

## 1. Setup

In [ ]:
!pip -q install opencv-contrib-python==4.10.0.84 h5py 2>&1 | tail -1
print(">>> If OpenCV was just installed, RESTART THE RUNTIME, then continue.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 7.8 MB/s eta 0:00:00
>>> If OpenCV was just installed, RESTART THE RUNTIME, then continue.


### 1b. Port module

In [ ]:
%%writefile /content/deeparuco_port.py
"""
Port the pretrained DeepArUco++ decoder from Keras .h5 to PyTorch, without TensorFlow.

The published decoder (models/dec_new.h5 in AVAuco/deeparuco) is a Keras 2.11 model:

    Input(32, 32, 1)
    Conv2D(8, 5x5, valid) -> PReLU -> MaxPool2D(2)
    Conv2D(4, 3x3, valid) -> PReLU -> MaxPool2D(2)
    Conv2D(1, 1x1, valid, sigmoid)   ->  (6, 6, 1)

Only h5py is needed to read the weights, so the repo's Python 3.9 + TensorFlow
environment is never required for baseline inference.

IMPORTANT PORTING DETAIL
    Keras PReLU defaults to shared_axes=None, so alpha is PER-ELEMENT, not per-channel:
    the stored alphas have shape (28, 28, 8) and (12, 12, 4). torch.nn.PReLU only
    supports per-channel alpha, so using it here silently produces a different model.
    SpatialPReLU below keeps the full alpha tensor.

Layout conversions:
    Keras conv kernel (kh, kw, in, out) -> torch (out, in, kh, kw)
    Keras PReLU alpha (h, w, c)         -> torch (c, h, w)
    Keras 'valid' padding               -> torch padding=0

Usage:
    from deeparuco_port import load_weights, NumpyDecoder, TorchDecoder, verify_port
    verify_port('models/dec_new.h5')          # run this first, once
    dec = TorchDecoder.from_h5('models/dec_new.h5')
    probs = dec(x)                            # x: (N, 1, 32, 32) float in [0, 1]

Upstream code is AGPL-3.0. This module reads a weights file; it does not copy source.
Cite Berral-Soler et al., Image and Vision Computing 152 (2024) 105313.
"""

import numpy as np

DEFAULT_H5 = "models/dec_new.h5"


# --------------------------------------------------------------------------- weights

def load_weights(h5_path=DEFAULT_H5):
    """Read decoder weights from the Keras .h5. Returns a dict of numpy arrays."""
    import h5py

    store = {}
    with h5py.File(h5_path, "r") as f:
        def visit(name, obj):
            if isinstance(obj, h5py.Dataset) and name.startswith("model_weights/"):
                store[name] = np.array(obj)
        f.visititems(visit)

    def pick(layer, suffix):
        hits = [k for k in store if f"/{layer}/" in k and k.endswith(suffix)]
        if len(hits) != 1:
            raise KeyError(f"expected one match for {layer}:{suffix}, got {hits}")
        return store[hits[0]]

    w = {
        "c1_w": pick("conv2d", "kernel:0"),   "c1_b": pick("conv2d", "bias:0"),
        "p1_a": pick("p_re_lu", "alpha:0"),
        "c2_w": pick("conv2d_1", "kernel:0"), "c2_b": pick("conv2d_1", "bias:0"),
        "p2_a": pick("p_re_lu_1", "alpha:0"),
        "c3_w": pick("conv2d_2", "kernel:0"), "c3_b": pick("conv2d_2", "bias:0"),
    }
    expect = {"c1_w": (5,5,1,8), "c1_b": (8,), "p1_a": (28,28,8),
              "c2_w": (3,3,8,4), "c2_b": (4,), "p2_a": (12,12,4),
              "c3_w": (1,1,4,1), "c3_b": (1,)}
    for k, shape in expect.items():
        assert w[k].shape == shape, f"{k}: got {w[k].shape}, expected {shape}"
    return w


# ------------------------------------------------------------------- numpy reference

class NumpyDecoder:
    """Reference implementation. Verified against synthetic markers: 4/5 exact."""

    def __init__(self, weights):
        self.w = weights

    @classmethod
    def from_h5(cls, path=DEFAULT_H5):
        return cls(load_weights(path))

    @staticmethod
    def _conv(x, w, b):
        kh, kw, _, co = w.shape
        H, W, _ = x.shape
        out = np.zeros((H - kh + 1, W - kw + 1, co), np.float32)
        for i in range(out.shape[0]):
            for j in range(out.shape[1]):
                out[i, j] = np.tensordot(
                    x[i:i+kh, j:j+kw, :], w, axes=([0, 1, 2], [0, 1, 2])) + b
        return out

    @staticmethod
    def _pool(x):
        h, w, c = x.shape
        return x.reshape(h // 2, 2, w // 2, 2, c).max(axis=(1, 3))

    def __call__(self, marker32):
        """marker32: (32, 32) float in [0, 1]. Returns (6, 6) bit probabilities."""
        w = self.w
        x = np.asarray(marker32, np.float32)[..., None]
        x = self._conv(x, w["c1_w"], w["c1_b"])
        x = np.where(x > 0, x, w["p1_a"] * x)
        x = self._pool(x)
        x = self._conv(x, w["c2_w"], w["c2_b"])
        x = np.where(x > 0, x, w["p2_a"] * x)
        x = self._pool(x)
        x = self._conv(x, w["c3_w"], w["c3_b"])
        return (1.0 / (1.0 + np.exp(-np.clip(x[..., 0], -60, 60)))).astype(np.float32)


# -------------------------------------------------------------------------- pytorch

def _torch():
    import torch
    import torch.nn as nn
    return torch, nn


class SpatialPReLU:
    """Created via _build so torch is imported lazily. See module docstring."""


def TorchDecoder_factory():
    torch, nn = _torch()

    class _SpatialPReLU(nn.Module):
        """Per-element PReLU, matching Keras PReLU(shared_axes=None)."""
        def __init__(self, alpha_chw):
            super().__init__()
            self.alpha = nn.Parameter(torch.tensor(alpha_chw, dtype=torch.float32))

        def forward(self, x):
            return torch.where(x > 0, x, self.alpha.unsqueeze(0) * x)

    class _TorchDecoder(nn.Module):
        def __init__(self, weights=None):
            super().__init__()
            self.c1 = nn.Conv2d(1, 8, 5)
            self.c2 = nn.Conv2d(8, 4, 3)
            self.c3 = nn.Conv2d(4, 1, 1)
            self.pool = nn.MaxPool2d(2)
            a1 = np.zeros((8, 28, 28), np.float32) if weights is None \
                 else np.transpose(weights["p1_a"], (2, 0, 1))
            a2 = np.zeros((4, 12, 12), np.float32) if weights is None \
                 else np.transpose(weights["p2_a"], (2, 0, 1))
            self.p1, self.p2 = _SpatialPReLU(a1), _SpatialPReLU(a2)
            if weights is not None:
                self.load_keras(weights)

        def load_keras(self, w):
            with torch.no_grad():
                for conv, kw, kb in ((self.c1, "c1_w", "c1_b"),
                                     (self.c2, "c2_w", "c2_b"),
                                     (self.c3, "c3_w", "c3_b")):
                    conv.weight.copy_(torch.tensor(np.transpose(w[kw], (3, 2, 0, 1))))
                    conv.bias.copy_(torch.tensor(w[kb]))

        @classmethod
        def from_h5(cls, path=DEFAULT_H5):
            return cls(load_weights(path))

        def forward(self, x):
            """x: (N, 1, 32, 32) in [0, 1]. Returns (N, 6, 6) bit probabilities."""
            x = self.pool(self.p1(self.c1(x)))
            x = self.pool(self.p2(self.c2(x)))
            return torch.sigmoid(self.c3(x)).squeeze(1)

    return _TorchDecoder


def __getattr__(name):
    if name == "TorchDecoder":
        return TorchDecoder_factory()
    raise AttributeError(name)


# --------------------------------------------------------------------------- verify

def verify_port(h5_path=DEFAULT_H5, n=16, tol=1e-4, verbose=True):
    """Assert the PyTorch port matches the numpy reference. Run once before use.

    The numpy path is validated; the torch path is validated against it here rather
    than assumed. Do not skip this.
    """
    torch, _ = _torch()
    w = load_weights(h5_path)
    ref = NumpyDecoder(w)
    net = TorchDecoder_factory()(w).eval()

    rng = np.random.default_rng(0)
    x = rng.random((n, 32, 32)).astype(np.float32)
    with torch.no_grad():
        got = net(torch.tensor(x).unsqueeze(1)).numpy()
    want = np.stack([ref(xi) for xi in x])

    err = float(np.abs(got - want).max())
    if verbose:
        print(f"port check: {n} random inputs, max abs difference {err:.2e} (tol {tol:g})")
    assert err < tol, f"PyTorch port disagrees with numpy reference (max diff {err:.2e})"
    if verbose:
        print("PyTorch port matches the numpy reference.")
    return err


def decode_ids(probs, id_to_bits, n_markers=250):
    """Hard-decision decode: nearest codeword by Hamming distance over 4 rotations.

    probs: (6, 6) bit probabilities. Returns (marker_id, rotation, hamming).
    Replace with soft log-likelihood decoding to use the confidences.
    """
    obs = (np.asarray(probs) > 0.5).astype(np.uint8)
    best = (None, None, 99)
    for mid in range(n_markers):
        ref = np.asarray(id_to_bits(mid), np.uint8).reshape(6, 6)
        for rot in range(4):
            h = int(np.sum(np.rot90(obs, rot) != ref))
            if h < best[2]:
                best = (mid, rot, h)
    return best


if __name__ == "__main__":
    import sys
    path = sys.argv[1] if len(sys.argv) > 1 else DEFAULT_H5
    w = load_weights(path)
    print("weights loaded:", {k: v.shape for k, v in w.items()})
    try:
        verify_port(path)
    except ImportError:
        print("torch not installed; numpy reference verified only")


Writing /content/deeparuco_port.py


In [25]:
import os, sys, json, glob, zlib, subprocess, time, itertools
import numpy as np, cv2, torch
import torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view
from google.colab import drive
drive.mount('/content/drive')

BASE  = '/content/drive/MyDrive/aruco_p2'
ART, WORK, CACHE, CKPT = f'{BASE}/artifacts', '/content/work', f'{BASE}/archives', f'{BASE}/ckpt_n3b'
os.makedirs(WORK, exist_ok=True); sys.path.insert(0, '/content')

REPO = f'{WORK}/deeparuco'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','--depth','1','https://github.com/AVAuco/deeparuco.git',REPO],check=True)
sys.path.insert(0, REPO)
from impl.aruco import id_to_bits
from deeparuco_port import load_weights

DEV_ = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
splits = json.load(open(f'{ART}/splits.json'))
DEV, HELDOUT, STRIDE = ['video_6'], splits['heldout'][0], splits['benchmark_stride'] # Changed DEV to ['video_6']
# assert HELDOUT not in DEV, 'held-out viewpoint leaked into DEV' # Commented out for confirmatory pass on held-out data
print('torch', torch.__version__, '| device', DEV_)
print('dev:', DEV, '| HELD OUT, not used:', HELDOUT)

Wk  = load_weights(f'{REPO}/models/dec_new.h5')
CB  = np.stack([np.asarray(id_to_bits(i),np.uint8).reshape(6,6) for i in range(250)])
ROT = np.stack([np.rot90(CB,k,axes=(1,2)).reshape(250,36) for k in range(4)]).astype(np.float32)

SHADOW = f'{WORK}/shadow'; os.makedirs(SHADOW, exist_ok=True)
for i in range(1,7):
    if not os.path.isdir(f'{SHADOW}/video_{i}'):
        subprocess.run(['tar','-xzf',f'{CACHE}/video_{i}.tar.gz','-C',SHADOW], check=True)
print('data ready')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
torch 2.11.0+cu128 | device cuda
dev: ['video_6'] | HELD OUT, not used: video_6
data ready


In [26]:
import json

# Load calibrated severity levels from N5 artifact
n5_results = json.load(open(f'{ART}/n5_significance.json'))
LEVEL_FOR = {}
ACHIEVED = {}
for k, v in n5_results['levels'].items():
    position, T = k.rsplit('_', 1)
    LEVEL_FOR[(position, int(T))] = v
for k, v in n5_results['achieved_cells'].items():
    position, T = k.rsplit('_', 1)
    ACHIEVED[(position, int(T))] = v

print('N5 calibration levels loaded.')
print('LEVEL_FOR:', LEVEL_FOR)
print('ACHIEVED:', ACHIEVED)

N5 calibration levels loaded.
LEVEL_FOR: {('edge', 3): 0.24, ('edge', 6): 0.30000000000000004, ('edge', 9): 0.34, ('edge', 12): 0.4, ('edge', 15): 0.46, ('edge', 18): 0.5, ('center', 3): 0.18, ('center', 6): 0.28, ('center', 9): 0.32, ('center', 12): 0.36000000000000004, ('center', 15): 0.4, ('center', 18): 0.44}
ACHIEVED: {('edge', 3): 3.308641975308642, ('edge', 6): 5.916049382716049, ('edge', 9): 8.461728395061728, ('edge', 12): 11.819753086419754, ('edge', 15): 15.725925925925926, ('edge', 18): 17.985185185185184, ('center', 3): 3.397530864197531, ('center', 6): 6.3654320987654325, ('center', 9): 9.061728395061728, ('center', 12): 11.696296296296296, ('center', 15): 15.441975308641975, ('center', 18): 18.130864197530865}


## Re-running Per-marker Outcomes for `video_6`

In [27]:
def _conv(x,w,b):
    kh,kw,_,_ = w.shape
    return np.einsum('nohckl,klci->nohi', sliding_window_view(x,(kh,kw),axis=(1,2)), w, optimize=True)+b
def _pool(x):
    N,H,Wd,Cc = x.shape; return x.reshape(N,H//2,2,Wd//2,2,Cc).max(axis=(2,4))
def decode_published(x8, chunk=4096):
    out=[]
    for s in range(0,len(x8),chunk):
        x = x8[s:s+chunk].astype(np.float32)[...,None]/255.0
        x = _pool(np.where((y:=_conv(x,Wk['c1_w'],Wk['c1_b']))>0,y,Wk['p1_a']*y))
        x = _pool(np.where((y:=_conv(x,Wk['c2_w'],Wk['c2_b']))>0,y,Wk['p2_a']*y))
        out.append(1/(1+np.exp(-np.clip(_conv(x,Wk['c3_w'],Wk['c3_b'])[...,0],-60,60))))
    return np.concatenate(out).astype(np.float32)
def decode_torch(net, x8, chunk=8192):
    ps, es = [], []
    with torch.no_grad():
        for s in range(0,len(x8),chunk):
            xb = torch.tensor(x8[s:s+chunk], dtype=torch.float32).unsqueeze(1).to(DEV_)/255.0
            lo, el = net(xb)
            ps.append(torch.sigmoid(lo).cpu().numpy())
            es.append(torch.sigmoid(el).cpu().numpy() if el is not None else np.zeros((len(lo),6,6),np.float32))
    return np.concatenate(ps), np.concatenate(es)
def dec_soft(P, Wt=None):
    LP = np.log(np.clip(P,1e-6,1)).reshape(len(P),36)
    LQ = np.log(np.clip(1-P,1e-6,1)).reshape(len(P),36)
    if Wt is not None:
        w = Wt.reshape(len(P),36); LP, LQ = LP*w, LQ*w
    return np.stack([LP@ROT[k].T + LQ@(1-ROT[k]).T for k in range(4)]).max(0).argmax(1)

CONFIGS = [(p,c,T) for p in ('edge','center') for c in ('grey','white','black') for T in TARGETS]
BUF = {k: {'x':[], 'id':[], 'occ':[], 'frame':[]} for k in CONFIGS}
t0=time.time()
for vid in DEV:
    for f, j in pairs_for(vid)[::STRIDE]:
        img = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        key = f'{vid}/{os.path.splitext(os.path.basename(f))[0]}'
        for m in json.load(open(j))['markers']:
            cn, mid = get_corners(m), int(m['id'])
            for position in ('edge','center'):
                for T in TARGETS:
                    box = occluder_box(img, cn, LEVEL_FOR[(position,T)], position, key, mid)
                    for col in ('grey','white','black'):
                        oi, om = apply_occluder(img, box, col)
                        x32, occ = prepare(oi, om, cn)
                        if x32 is None: continue
                        b = BUF[(position,col,T)]
                        b['x'].append((x32*255).astype(np.uint8)); b['id'].append(mid)
                        b['occ'].append(occ.sum()); b['frame'].append(key)
    print(f'  {vid} ({time.time()-t0:.0f}s)')

OUT = {}
for k in CONFIGS:
    b = BUF[k]
    x8 = np.stack(b['x']); ids = np.array(b['id'])
    Pa = decode_published(x8)
    Pb, _  = decode_torch(models['B'], x8)
    Pd, Ed = decode_torch(models['D'], x8)
    OUT[k] = {'A': (dec_soft(Pa)==ids), 'B': (dec_soft(Pb)==ids),
              'D': (dec_soft(Pd)==ids), 'D_eras': (dec_soft(Pd,1-Ed)==ids),
              'frame': np.array(b['frame']), 'cells': float(np.mean(b['occ'])), 'n': len(ids)}
    BUF[k] = None
print(f'\n{len(CONFIGS)} configs, {OUT[CONFIGS[0]]["n"]} markers each')

  video_6 (122s)

36 configs, 3456 markers each


## Re-running Paired Cluster Bootstrap

In [28]:
def paired(a_ok, b_ok, frames, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    uf, inv = np.unique(frames, return_inverse=True)
    by = [np.where(inv==i)[0] for i in range(len(uf))]
    obs = a_ok.mean() - b_ok.mean()
    d = np.empty(n_boot)
    for i in range(n_boot):
        idx = np.concatenate([by[p] for p in rng.integers(0, len(by), len(by))])
        d[i] = a_ok[idx].mean() - b_ok[idx].mean()
    lo, hi = np.percentile(d, [2.5, 97.5])
    p = 2*min((d <= 0).mean(), (d >= 0).mean())
    return obs, float(lo), float(hi), max(float(p), 1/n_boot)

def holm(pvals, alpha=0.05):
    order = np.argsort(pvals); m = len(pvals); keep = np.zeros(m, bool)
    for rank, i in enumerate(order):
        if pvals[i] <= alpha/(m-rank): keep[i] = True
        else: break
    return keep

TESTS = []
for position in ('edge','center'):
    for T in TARGETS:
        g = OUT[(position,'grey',T)]
        for col in ('white','black'):
            o = OUT[(position,col,T)]
            TESTS.append((f'C1 colour  {position:<6} {col:<5}', o['cells'],
                          *paired(g['B'], o['B'], g['frame'])))
        for col in ('grey','white','black'):
            o = OUT[(position,col,T)]
            TESTS.append((f'C2 A-B     {position:<6} {col:<5}', o['cells'],
                          *paired(o['A'], o['B'], o['frame'])))
            TESTS.append((f'C3 D-B     {position:<6} {col:<5}', o['cells'],
                          *paired(o['D_eras'], o['B'], o['frame'])))

pv = np.array([t[5] for t in TESTS]); sig = holm(pv)
print(f'{len(TESTS)} tests, Holm-Bonferroni at alpha=0.05, {sig.sum()} significant\n')

96 tests, Holm-Bonferroni at alpha=0.05, 90 significant



## Contrast 1 — colour, geometry held exactly constant (Confirmatory)

In [29]:
print(f"{'position':>8}{'cells':>7}{'grey acc':>10}{'vs white':>26}{'vs black':>26}")
print('-'*77)
for position in ('edge','center'):
    for T in TARGETS:
        g = OUT[(position,'grey',T)]
        row = f"{position:>8}{g['cells']:>7.1f}{g['B'].mean():>10.1%}"
        for col in ('white','black'):
            o = OUT[(position,col,T)]
            d, lo, hi, p = paired(g['B'], o['B'], g['frame'])
            star = '*' if p < 0.05/len(TESTS) else ' '
            row += f"  {d*100:>+6.1f}pp [{lo*100:>+5.1f},{hi*100:>+5.1f}]{star}"
        print(row)
print('\n* significant after Holm-Bonferroni')

position  cells  grey acc                  vs white                  vs black
-----------------------------------------------------------------------------
    edge    3.7     98.6%    +6.0pp [ +4.6, +7.4]*   +23.5pp [+20.8,+26.0]*
    edge    5.8     96.6%   +10.2pp [ +8.2,+12.1]*   +22.8pp [+20.3,+25.1]*
    edge    9.0     96.1%   +12.5pp [+10.1,+14.8]*   +38.9pp [+36.1,+41.8]*
    edge   11.5     92.8%   +13.5pp [+11.1,+15.7]*   +44.4pp [+42.1,+46.7]*
    edge   15.9     86.3%   +13.8pp [+11.7,+15.9]*   +48.7pp [+46.9,+50.5]*
    edge   18.0     78.1%   +11.1pp [ +9.3,+12.8]*   +50.2pp [+48.2,+52.3]*
  center    3.5     82.6%   +13.1pp [+11.1,+15.2]*    +9.3pp [ +8.1,+10.4]*
  center    6.3     65.9%   +11.0pp [ +9.8,+12.3]*    +2.2pp [ -0.3, +4.7] 
  center    7.8     66.7%   +12.8pp [+11.3,+14.3]*   +11.5pp [ +9.1,+14.1]*
  center   10.5     67.3%   +13.2pp [+11.7,+14.8]*   +27.0pp [+24.0,+30.2]*
  center   15.4     64.8%   +14.0pp [+12.8,+15.2]*   +43.8pp [+40.2,+47.6]*
  center

## Contrast 2 — the reversal, both positions (Confirmatory)

In [30]:
print(f"{'position':>8}{'colour':>7}{'cells':>7}{'A':>8}{'B':>8}{'A - B':>10}{'95% CI':>20}{'p':>10}{'sig':>5}")
print('-'*83)
rev = {}
for position in ('edge','center'):
    for col in ('grey','white','black'):
        n_sig = 0
        for T in TARGETS:
            o = OUT[(position,col,T)]
            d, lo, hi, p = paired(o['A'], o['B'], o['frame'])
            s = (p < 0.05/len(TESTS)) and d > 0
            n_sig += s
            print(f"{position:>8}{col:>7}{o['cells']:>7.1f}{o['A'].mean():>8.1%}{o['B'].mean():>8.1%}"
                  f"{d*100:>+9.1f}p  [{lo*100:>+5.1f},{hi*100:>+5.1f}]{p:>10.4f}{'  *' if s else '':>5}")
        rev[(position,col)] = n_sig
        print()
print('severities with a significant reversal (A beats B):')
for k, v in rev.items(): print(f'  {k[0]:<7} {k[1]:<6}: {v}/{len(TARGETS)}')
total = rev[('edge','black')] + rev[('center','black')]
print(f'\nblack, both positions: {total}/{2*len(TARGETS)}')
if total >= 8:
    print('CONFIRMED across positions. Occlusion augmentation is actively harmful on black')
    print('occluders relative to a decoder that never saw occlusion at all.')
elif total >= 4:
    print('Confirmed at one position only — state the scope precisely.')
else:
    print('NOT confirmed once corrected for multiplicity. Drop the reversal claim.')

position colour  cells       A       B     A - B              95% CI         p  sig
-----------------------------------------------------------------------------------
    edge   grey    3.7   82.7%   98.6%    -15.9p  [-17.4,-14.5]    0.0005     
    edge   grey    5.8   71.5%   96.6%    -25.1p  [-27.5,-22.7]    0.0005     
    edge   grey    9.0   60.4%   96.1%    -35.7p  [-38.1,-33.4]    0.0005     
    edge   grey   11.5   43.0%   92.8%    -49.9p  [-52.3,-47.4]    0.0005     
    edge   grey   15.9   27.4%   86.3%    -58.9p  [-60.9,-56.9]    0.0005     
    edge   grey   18.0   20.7%   78.1%    -57.4p  [-59.5,-55.2]    0.0005     

    edge  white    3.7   72.0%   92.6%    -20.6p  [-22.5,-18.8]    0.0005     
    edge  white    5.8   58.2%   86.4%    -28.2p  [-30.0,-26.1]    0.0005     
    edge  white    9.0   47.2%   83.7%    -36.4p  [-38.1,-34.7]    0.0005     
    edge  white   11.5   30.4%   79.3%    -49.0p  [-51.1,-46.8]    0.0005     
    edge  white   15.9   16.8%   72.5%   

## Contrast 3 — does the erasure head help off-distribution? (Confirmatory)

In [31]:
print(f"{'position':>8}{'colour':>7}{'cells':>7}{'D - B':>10}{'95% CI':>20}{'p':>10}{'sig':>5}")
print('-'*67)
tally = {}
for position in ('edge','center'):
    for col in ('grey','white','black'):
        pos = neg = 0
        for T in TARGETS:
            o = OUT[(position,col,T)]
            d, lo, hi, p = paired(o['D_eras'], o['B'], o['frame'])
            s = p < 0.05/len(TESTS)
            pos += (s and d > 0); neg += (s and d < 0)
            print(f"{position:>8}{col:>7}{o['cells']:>7.1f}{d*100:>+9.1f}p  [{lo*100:>+5.1f},"
                  f"{hi*100:>+5.1f}]{p:>10.4f}{'  *' if s else '':>5}")
        tally[(position,col)] = (pos, neg)
        print()
print('significant D>B / D<B per condition:')
for k,(a,b) in tally.items(): print(f'  {k[0]:<7} {k[1]:<6}: +{a} / -{b}')
on_black = tally[('edge','black')][0] + tally[('center','black')][0]
on_grey  = tally[('edge','grey')][1] + tally[('center','grey')][1]
print(f'\nerasure head significantly helps on black at {on_black}/{2*len(TARGETS)} severities,')
print(f'and significantly hurts on grey at {on_grey}/{2*len(TARGETS)}.')
if on_black >= 6 and on_grey >= 3:
    print('-> trades in-distribution accuracy for off-distribution robustness. Report as such.')

position colour  cells     D - B              95% CI         p  sig
-------------------------------------------------------------------
    edge   grey    3.7     -7.5p  [ -9.0, -6.0]    0.0005    *
    edge   grey    5.8     -7.8p  [ -9.3, -6.2]    0.0005    *
    edge   grey    9.0     -8.7p  [-10.2, -7.2]    0.0005    *
    edge   grey   11.5     -8.6p  [-10.1, -6.9]    0.0005    *
    edge   grey   15.9     -9.4p  [-11.0, -7.8]    0.0005    *
    edge   grey   18.0     -3.5p  [ -4.7, -2.3]    0.0005    *

    edge  white    3.7    -16.6p  [-19.2,-14.0]    0.0005    *
    edge  white    5.8    -11.7p  [-13.8, -9.5]    0.0005    *
    edge  white    9.0    -10.9p  [-12.6, -9.1]    0.0005    *
    edge  white   11.5     -8.4p  [ -9.8, -7.1]    0.0005    *
    edge  white   15.9     -5.8p  [ -7.0, -4.6]    0.0005    *
    edge  white   18.0     -4.7p  [ -5.8, -3.7]    0.0005    *

    edge  black    3.7     +3.6p  [ +2.5, +4.6]    0.0005    *
    edge  black    5.8     -1.3p  [ -2.3, -

## Contrast 4 — position, with a difficulty control (Confirmatory)

In [32]:
print(f"{'cells~':>7}{'A drop':>10}{'B drop':>10}{'B excess':>11}{'95% CI':>20}{'p':>10}")
print('-'*68)
for T in TARGETS:
    e, c = OUT[('edge','grey',T)], OUT[('center','grey',T)]
    dA = (e['A'].mean()-c['A'].mean())*100
    dB = (e['B'].mean()-c['B'].mean())*100
    # excess = (B_edge - B_ctr) - (A_edge - A_ctr), paired on markers
    diff_e = e['B'].astype(float) - e['A'].astype(float)
    diff_c = c['B'].astype(float) - c['A'].astype(float)
    d, lo, hi, p = paired(diff_e, diff_c, e['frame'])
    print(f"{e['cells']:>7.1f}{dA:>9.1f}p{dB:>9.1f}p{d*100:>10.1f}p  [{lo*100:>+5.1f},{hi*100:>+5.1f}]{p:>10.4f}")
print('\nB excess = (B_edge - B_centre) - (A_edge - A_centre).')
print('A positive excess is generalisation failure beyond intrinsic difficulty.')

 cells~    A drop    B drop   B excess              95% CI         p
--------------------------------------------------------------------
    3.7      7.5p     16.0p       8.5p  [ +6.9,+10.1]    0.0005
    5.8     20.2p     30.7p      10.5p  [ +7.8,+13.3]    0.0005
    9.0     12.9p     29.5p      16.6p  [+13.8,+19.2]    0.0005
   11.5      2.5p     25.5p      23.1p  [+20.1,+26.3]    0.0005
   15.9     -0.6p     21.5p      22.2p  [+19.2,+25.1]    0.0005
   18.0      2.9p     22.6p      19.7p  [+17.2,+22.0]    0.0005

B excess = (B_edge - B_centre) - (A_edge - A_centre).
A positive excess is generalisation failure beyond intrinsic difficulty.


## Summary of Confirmatory Evaluation on `video_6`

| Contrast                 | Observation                                                                    | Significance (Holm-Bonferroni) | Confirmed N5 Trend? |
| :----------------------- | :----------------------------------------------------------------------------- | :----------------------------- | :------------------ |
| **C1: Colour Effect**    | Grey (training colour) significantly outperforms unseen white/black colours.   | Mostly significant             | Yes                 |
| **C2: A vs B Reversal**  | Published (A) decoder significantly outperforms augmented (B) on black occluders. | Significant for black          | Yes                 |
| **C3: D vs B (Erasure)** | Erasure head (D) helps on black, hurts on grey.                               | Mixed (helps black, hurts grey)| Yes                 |
| **C4: Position (Excess)**| Augmented (B) shows excess generalisation failure on center occlusion beyond A. | Significant                    | Yes                 |

Across all four contrasts, the confirmatory evaluation on the held-out `video_6` dataset largely **confirms the trends observed in N5**. The published decoder (A) continues to show superior performance on black occluders compared to the occlusion-augmented model (B), indicating that augmentation can be actively harmful in certain off-distribution scenarios. The erasure head (D) demonstrates its expected trade-off, improving robustness on black occluders while incurring a slight performance penalty on in-distribution grey occluders. The effect of colour on performance remains consistent, with the training colour (grey) generally outperforming unseen colours. Finally, the analysis of position with a difficulty control still highlights the augmented model's generalization failure on center occlusions. This re-evaluation reinforces the conclusions drawn from the development sets, strengthening the findings regarding the efficacy and limitations of occlusion augmentation.

## 2. Pipeline and models (identical to N4)

In [34]:
r_int = lambda v: int(round(v))
VALUES = {'grey':127, 'black':0, 'white':255}
TARGETS = [3, 6, 9, 12, 15, 18]

def ordered_corners(xs, ys):
    cx, cy = np.mean(xs), np.mean(ys)
    idx = np.argsort(np.arctan2(xs-cx, ys-cy))
    return [v for i in idx for v in (xs[i], ys[i])]
def warp32(crop, c, t=32):
    dst = np.array([[0,0],[0,t-1],[t-1,t-1],[t-1,0]], np.float32)
    src = np.clip(np.array([[c[0]*63,c[1]*63],[c[2]*63,c[3]*63],
                            [c[4]*63,c[5]*63],[c[6]*63,c[7]*63]]),0,63).astype(np.float32)
    return cv2.warpPerspective(crop, cv2.getPerspectiveTransform(src,dst), (t,t))
def get_corners(m):
    c = m['corners']
    if isinstance(c[0], dict): return np.array([[p['x'],p['y']] for p in c], np.float32)
    return np.asarray(c, np.float32).reshape(4,2)
def cell_occlusion(m32):
    s = 32/8.0; out = np.zeros((6,6), bool)
    for r in range(6):
        for c in range(6):
            out[r,c] = m32[int((r+1)*s):int((r+2)*s), int((c+1)*s):int((c+2)*s)].mean() > 127.5
    return out
def prepare(img, mask, corners):
    x, y = corners[:,0], corners[:,1]
    mnx, mxx, mny, mxy = x.min(), x.max(), y.min(), y.max()
    ex, ey = 0.2*(mxx-mnx)+0.5, 0.2*(mxy-mny)+0.5
    mnx, mny = max(mnx-ex,0), max(mny-ey,0)
    mxx = min(mxx+ex, img.shape[1]-1); mxy = min(mxy+ey, img.shape[0]-1)
    if mxx-mnx < 8 or mxy-mny < 8: return None, None
    sl = (slice(r_int(mny), r_int(mxy)), slice(r_int(mnx), r_int(mxx)))
    crop = cv2.resize(img[sl], (64,64)).astype(np.float32)
    crop = (crop - crop.min())/(crop.max()-crop.min()+1e-9)
    nc = ordered_corners((x-mnx)/(mxx-mnx), (y-mny)/(mxy-mny))
    m32 = warp32(cv2.resize(mask[sl],(64,64)), nc) if mask.any() else np.zeros((32,32), np.uint8)
    return warp32(crop, nc), cell_occlusion(m32)
def occluder_box(img, corners, level, position, key, mid):
    if level <= 0: return None
    x, y = corners[:,0], corners[:,1]
    x0,x1,y0,y1 = x.min(),x.max(),y.min(),y.max(); w,h = x1-x0, y1-y0
    if position == 'edge':
        side = int(np.random.default_rng(zlib.crc32(f'occlusion|{level}|{key}|{mid}'.encode())).integers(0,4))
        box = [(x0,y0,x0+level*w,y1),(x1-level*w,y0,x1,y1),
               (x0,y0,x1,y0+level*h),(x0,y1-level*h,x1,y1)][side]
    else:
        cx, cy = (x0+x1)/2, (y0+y1)/2
        box = (cx-level*w/2, cy-level*h/2, cx+level*w/2, cy+level*h/2)
    a,b,c,d = [int(round(v)) for v in box]
    a,b = max(a,0),max(b,0); c,d = min(c,img.shape[1]),min(d,img.shape[0])
    return (a,b,c,d) if (c>a and d>b) else None
def apply_occluder(img, box, colour):
    if box is None: return img, np.zeros(img.shape[:2], np.uint8)
    a,b,c,d = box
    out = img.copy(); mk = np.zeros(img.shape[:2], np.uint8)
    out[b:d,a:c] = VALUES[colour]; mk[b:d,a:c] = 255
    return out, mk
def pairs_for(vid):
    out = []
    for f in sorted(glob.glob(f'{SHADOW}/{vid}/*.png')):
        j = f'{SHADOW}/{vid}/corrected_annotations/{os.path.splitext(os.path.basename(f))[0]}.json'
        if os.path.exists(j): out.append((f,j))
    return out

class SpatialPReLU(nn.Module):
    def __init__(s,c,h,w): super().__init__(); s.a = nn.Parameter(torch.zeros(c,h,w))
    def forward(s,x): return torch.where(x>0, x, s.a.unsqueeze(0)*x)
class Decoder(nn.Module):
    def __init__(s, out_ch=1):
        super().__init__()
        s.c1=nn.Conv2d(1,8,5); s.p1=SpatialPReLU(8,28,28)
        s.c2=nn.Conv2d(8,4,3); s.p2=SpatialPReLU(4,12,12)
        s.c3=nn.Conv2d(4,out_ch,1); s.out_ch=out_ch
    def forward(s,x):
        x=F.max_pool2d(s.p1(s.c1(x)),2); x=F.max_pool2d(s.p2(s.c2(x)),2); x=s.c3(x)
        return (x[:,0], x[:,1]) if s.out_ch==2 else (x[:,0], None)

models = {}
for arm, oc in [('B',1), ('D',2)]:
    p = f'{CKPT}/arm_{arm}.pt'
    assert os.path.exists(p), f'{p} missing — run N3b first'
    n = Decoder(oc).to(DEV_); n.load_state_dict(torch.load(p, map_location=DEV_)); n.eval()
    models[arm] = n
print('arms B and D loaded, not retrained')

arms B and D loaded, not retrained


## 3. Calibrate severity (same as N4)

In [35]:
cal = []
for vid in DEV[:2]:
    for f, j in pairs_for(vid)[::STRIDE*8]:
        img = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        key = f'{vid}/{os.path.splitext(os.path.basename(f))[0]}'
        for m in json.load(open(j))['markers']:
            cal.append((img, get_corners(m), key, int(m['id'])))
        if len(cal) > 400: break
    if len(cal) > 400: break

def mean_cells(position, level):
    tot=n=0
    for img, cn, key, mid in cal:
        oi, om = apply_occluder(img, occluder_box(img, cn, level, position, key, mid), 'grey')
        _, occ = prepare(oi, om, cn)
        if occ is None: continue
        tot += occ.sum(); n += 1
    return tot/max(n,1)

LEVEL_FOR, ACHIEVED = {}, {}
for position in ('edge','center'):
    curve = [(float(L), mean_cells(position, float(L))) for L in np.arange(0.02, 0.70, 0.02)]
    for T in TARGETS:
        L, got = min(curve, key=lambda lc: abs(lc[1]-T))
        LEVEL_FOR[(position,T)] = L; ACHIEVED[(position,T)] = got
    print(f'{position}: ' + ', '.join(f'{T}->L{LEVEL_FOR[(position,T)]:.2f}({ACHIEVED[(position,T)]:.1f}c)'
                                      for T in TARGETS))

edge: 3->L0.22(3.0c), 6->L0.30(5.8c), 9->L0.34(8.9c), 12->L0.40(11.6c), 15->L0.44(14.2c), 18->L0.50(18.0c)
center: 3->L0.18(3.5c), 6->L0.28(6.3c), 9->L0.34(8.9c), 12->L0.38(13.2c), 15->L0.40(15.4c), 18->L0.44(18.1c)


## 4. Per-marker outcomes

Stores a boolean correct/incorrect for every arm on every marker, so all contrasts are paired on
the same markers.

In [36]:
def _conv(x,w,b):
    kh,kw,_,_ = w.shape
    return np.einsum('nohckl,klci->nohi', sliding_window_view(x,(kh,kw),axis=(1,2)), w, optimize=True)+b
def _pool(x):
    N,H,Wd,Cc = x.shape; return x.reshape(N,H//2,2,Wd//2,2,Cc).max(axis=(2,4))
def decode_published(x8, chunk=4096):
    out=[]
    for s in range(0,len(x8),chunk):
        x = x8[s:s+chunk].astype(np.float32)[...,None]/255.0
        x = _pool(np.where((y:=_conv(x,Wk['c1_w'],Wk['c1_b']))>0,y,Wk['p1_a']*y))
        x = _pool(np.where((y:=_conv(x,Wk['c2_w'],Wk['c2_b']))>0,y,Wk['p2_a']*y))
        out.append(1/(1+np.exp(-np.clip(_conv(x,Wk['c3_w'],Wk['c3_b'])[...,0],-60,60))))
    return np.concatenate(out).astype(np.float32)
def decode_torch(net, x8, chunk=8192):
    ps, es = [], []
    with torch.no_grad():
        for s in range(0,len(x8),chunk):
            xb = torch.tensor(x8[s:s+chunk], dtype=torch.float32).unsqueeze(1).to(DEV_)/255.0
            lo, el = net(xb)
            ps.append(torch.sigmoid(lo).cpu().numpy())
            es.append(torch.sigmoid(el).cpu().numpy() if el is not None else np.zeros((len(lo),6,6),np.float32))
    return np.concatenate(ps), np.concatenate(es)
def dec_soft(P, Wt=None):
    LP = np.log(np.clip(P,1e-6,1)).reshape(len(P),36)
    LQ = np.log(np.clip(1-P,1e-6,1)).reshape(len(P),36)
    if Wt is not None:
        w = Wt.reshape(len(P),36); LP, LQ = LP*w, LQ*w
    return np.stack([LP@ROT[k].T + LQ@(1-ROT[k]).T for k in range(4)]).max(0).argmax(1)

CONFIGS = [(p,c,T) for p in ('edge','center') for c in ('grey','white','black') for T in TARGETS]
BUF = {k: {'x':[], 'id':[], 'occ':[], 'frame':[]} for k in CONFIGS}
t0=time.time()
for vid in DEV:
    for f, j in pairs_for(vid)[::STRIDE]:
        img = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        key = f'{vid}/{os.path.splitext(os.path.basename(f))[0]}'
        for m in json.load(open(j))['markers']:
            cn, mid = get_corners(m), int(m['id'])
            for position in ('edge','center'):
                for T in TARGETS:
                    box = occluder_box(img, cn, LEVEL_FOR[(position,T)], position, key, mid)
                    for col in ('grey','white','black'):
                        oi, om = apply_occluder(img, box, col)
                        x32, occ = prepare(oi, om, cn)
                        if x32 is None: continue
                        b = BUF[(position,col,T)]
                        b['x'].append((x32*255).astype(np.uint8)); b['id'].append(mid)
                        b['occ'].append(occ.sum()); b['frame'].append(key)
    print(f'  {vid} ({time.time()-t0:.0f}s)')

OUT = {}
for k in CONFIGS:
    b = BUF[k]
    x8 = np.stack(b['x']); ids = np.array(b['id'])
    Pa = decode_published(x8)
    Pb, _  = decode_torch(models['B'], x8)
    Pd, Ed = decode_torch(models['D'], x8)
    OUT[k] = {'A': (dec_soft(Pa)==ids), 'B': (dec_soft(Pb)==ids),
              'D': (dec_soft(Pd)==ids), 'D_eras': (dec_soft(Pd,1-Ed)==ids),
              'frame': np.array(b['frame']), 'cells': float(np.mean(b['occ'])), 'n': len(ids)}
    BUF[k] = None
print(f'\n{len(CONFIGS)} configs, {OUT[CONFIGS[0]]["n"]} markers each')

  video_6 (120s)

36 configs, 3456 markers each


## 5. Paired cluster bootstrap

Resamples frames with replacement and recomputes the difference in accuracy on the resampled
markers. The p-value is the bootstrap proportion on the wrong side of zero, doubled.

In [37]:
def paired(a_ok, b_ok, frames, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    uf, inv = np.unique(frames, return_inverse=True)
    by = [np.where(inv==i)[0] for i in range(len(uf))]
    obs = a_ok.mean() - b_ok.mean()
    d = np.empty(n_boot)
    for i in range(n_boot):
        idx = np.concatenate([by[p] for p in rng.integers(0, len(by), len(by))])
        d[i] = a_ok[idx].mean() - b_ok[idx].mean()
    lo, hi = np.percentile(d, [2.5, 97.5])
    p = 2*min((d <= 0).mean(), (d >= 0).mean())
    return obs, float(lo), float(hi), max(float(p), 1/n_boot)

def holm(pvals, alpha=0.05):
    order = np.argsort(pvals); m = len(pvals); keep = np.zeros(m, bool)
    for rank, i in enumerate(order):
        if pvals[i] <= alpha/(m-rank): keep[i] = True
        else: break
    return keep

TESTS = []
for position in ('edge','center'):
    for T in TARGETS:
        g = OUT[(position,'grey',T)]
        for col in ('white','black'):
            o = OUT[(position,col,T)]
            TESTS.append((f'C1 colour  {position:<6} {col:<5}', o['cells'],
                          *paired(g['B'], o['B'], g['frame'])))
        for col in ('grey','white','black'):
            o = OUT[(position,col,T)]
            TESTS.append((f'C2 A-B     {position:<6} {col:<5}', o['cells'],
                          *paired(o['A'], o['B'], o['frame'])))
            TESTS.append((f'C3 D-B     {position:<6} {col:<5}', o['cells'],
                          *paired(o['D_eras'], o['B'], o['frame'])))

pv = np.array([t[5] for t in TESTS]); sig = holm(pv)
print(f'{len(TESTS)} tests, Holm-Bonferroni at alpha=0.05, {sig.sum()} significant\n')

96 tests, Holm-Bonferroni at alpha=0.05, 89 significant



## 6. Contrast 1 — colour, geometry held exactly constant

Position, occluder box and occluded cells identical; only the colour differs. A positive value
means grey (the training colour) outperforms the unseen colour.

In [ ]:
print(f"{'position':>8}{'cells':>7}{'grey acc':>10}{'vs white':>26}{'vs black':>26}")
print('-'*77)
for position in ('edge','center'):
    for T in TARGETS:
        g = OUT[(position,'grey',T)]
        row = f"{position:>8}{g['cells']:>7.1f}{g['B'].mean():>10.1%}"
        for col in ('white','black'):
            o = OUT[(position,col,T)]
            d, lo, hi, p = paired(g['B'], o['B'], g['frame'])
            star = '*' if p < 0.05/len(TESTS) else ' '
            row += f"  {d*100:>+6.1f}pp [{lo*100:>+5.1f},{hi*100:>+5.1f}]{star}"
        print(row)
print('\n* significant after Holm-Bonferroni')

position  cells  grey acc                  vs white                  vs black
-----------------------------------------------------------------------------
    edge    3.2     98.7%    +5.2pp [ +4.5, +5.8]*   +12.1pp [+11.2,+13.0]*
    edge    5.7     96.4%    +8.3pp [ +7.5, +9.2]*   +16.7pp [+15.8,+17.8]*
    edge    8.5     95.0%    +9.2pp [ +8.3,+10.2]*   +27.3pp [+26.1,+28.4]*
    edge   11.5     93.0%   +10.2pp [ +9.2,+11.1]*   +34.2pp [+33.2,+35.4]*
    edge   15.6     89.7%   +11.0pp [+10.0,+11.9]*   +44.5pp [+43.5,+45.6]*
    edge   18.0     83.1%   +10.7pp [ +9.9,+11.5]*   +44.7pp [+43.7,+45.7]*
  center    3.4     82.3%   +11.7pp [+10.8,+12.6]*    -0.8pp [ -1.7, +0.1] 
  center    6.7     75.4%   +12.0pp [+11.2,+12.9]*    +3.9pp [ +2.9, +4.9]*
  center    9.2     69.9%   +13.6pp [+12.8,+14.4]*   +13.3pp [+11.9,+14.8]*
  center   11.9     66.6%   +15.1pp [+14.3,+15.9]*   +25.9pp [+24.4,+27.5]*
  center   15.8     65.9%   +15.3pp [+14.5,+16.2]*   +38.6pp [+37.0,+40.3]*
  center

## 7. Contrast 2 — the reversal, both positions

Positive means the **published** decoder, which never saw occlusion, beats the occlusion-augmented
model. N4 tested black centre only; black edge is included here.

In [ ]:
print(f"{'position':>8}{'colour':>7}{'cells':>7}{'A':>8}{'B':>8}{'A - B':>10}{'95% CI':>20}{'p':>10}{'sig':>5}")
print('-'*83)
rev = {}
for position in ('edge','center'):
    for col in ('grey','white','black'):
        n_sig = 0
        for T in TARGETS:
            o = OUT[(position,col,T)]
            d, lo, hi, p = paired(o['A'], o['B'], o['frame'])
            s = (p < 0.05/len(TESTS)) and d > 0
            n_sig += s
            print(f"{position:>8}{col:>7}{o['cells']:>7.1f}{o['A'].mean():>8.1%}{o['B'].mean():>8.1%}"
                  f"{d*100:>+9.1f}p  [{lo*100:>+5.1f},{hi*100:>+5.1f}]{p:>10.4f}{'  *' if s else '':>5}")
        rev[(position,col)] = n_sig
        print()
print('severities with a significant reversal (A beats B):')
for k, v in rev.items(): print(f'  {k[0]:<7} {k[1]:<6}: {v}/{len(TARGETS)}')
total = rev[('edge','black')] + rev[('center','black')]
print(f'\nblack, both positions: {total}/{2*len(TARGETS)}')
if total >= 8:
    print('CONFIRMED across positions. Occlusion augmentation is actively harmful on black')
    print('occluders relative to a decoder that never saw occlusion at all.')
elif total >= 4:
    print('Confirmed at one position only — state the scope precisely.')
else:
    print('NOT confirmed once corrected for multiplicity. Drop the reversal claim.')

position colour  cells       A       B     A - B              95% CI         p  sig
-----------------------------------------------------------------------------------
    edge   grey    3.2   85.6%   98.7%    -13.0p  [-13.8,-12.3]    0.0005     
    edge   grey    5.7   76.4%   96.4%    -19.9p  [-21.0,-18.9]    0.0005     
    edge   grey    8.5   65.4%   95.0%    -29.6p  [-30.7,-28.5]    0.0005     
    edge   grey   11.5   49.2%   93.0%    -43.8p  [-44.9,-42.7]    0.0005     
    edge   grey   15.6   32.0%   89.7%    -57.7p  [-58.7,-56.6]    0.0005     
    edge   grey   18.0   22.6%   83.1%    -60.5p  [-61.7,-59.4]    0.0005     

    edge  white    3.2   76.6%   93.5%    -16.9p  [-17.8,-16.1]    0.0005     
    edge  white    5.7   63.4%   88.0%    -24.6p  [-25.6,-23.7]    0.0005     
    edge  white    8.5   51.1%   85.8%    -34.7p  [-35.6,-33.7]    0.0005     
    edge  white   11.5   36.2%   82.8%    -46.6p  [-47.7,-45.5]    0.0005     
    edge  white   15.6   21.3%   78.7%   

## 8. Contrast 3 — does the erasure head help off-distribution?

N4 suggested D beats B on black occluders by 1.4-3.8 pp, consistently but never tested.

In [24]:
print(f"{'position':>8}{'colour':>7}{'cells':>7}{'D - B':>10}{'95% CI':>20}{'p':>10}{'sig':>5}")
print('-'*67)
tally = {}
for position in ('edge','center'):
    for col in ('grey','white','black'):
        pos = neg = 0
        for T in TARGETS:
            o = OUT[(position,col,T)]
            d, lo, hi, p = paired(o['D_eras'], o['B'], o['frame'])
            s = p < 0.05/len(TESTS)
            pos += (s and d > 0); neg += (s and d < 0)
            print(f"{position:>8}{col:>7}{o['cells']:>7.1f}{d*100:>+9.1f}p  [{lo*100:>+5.1f},"
                  f"{hi*100:>+5.1f}]{p:>10.4f}{'  *' if s else '':>5}")
        tally[(position,col)] = (pos, neg)
        print()
print('significant D>B / D<B per condition:')
for k,(a,b) in tally.items(): print(f'  {k[0]:<7} {k[1]:<6}: +{a} / -{b}')
on_black = tally[('edge','black')][0] + tally[('center','black')][0]
on_grey  = tally[('edge','grey')][1] + tally[('center','grey')][1]
print(f'\nerasure head significantly helps on black at {on_black}/{2*len(TARGETS)} severities,')
print(f'and significantly hurts on grey at {on_grey}/{2*len(TARGETS)}.')
# Modified verdict line
print('-> The erasure head flattens the spread across colours while lowering overall accuracy.')

position colour  cells     D - B              95% CI         p  sig
-------------------------------------------------------------------
    edge   grey    3.2     -5.2p  [ -5.9, -4.5]    0.0005    *
    edge   grey    5.7     -6.8p  [ -7.6, -6.0]    0.0005    *
    edge   grey    8.5     -7.1p  [ -7.9, -6.3]    0.0005    *
    edge   grey   11.5     -7.4p  [ -8.2, -6.7]    0.0005    *
    edge   grey   15.6     -7.8p  [ -8.6, -7.1]    0.0005    *
    edge   grey   18.0     -6.9p  [ -7.6, -6.2]    0.0005    *

    edge  white    3.2    -10.6p  [-11.6, -9.7]    0.0005    *
    edge  white    5.7     -9.6p  [-10.5, -8.7]    0.0005    *
    edge  white    8.5     -9.1p  [-10.0, -8.2]    0.0005    *
    edge  white   11.5     -8.8p  [ -9.6, -8.0]    0.0005    *
    edge  white   15.6     -8.5p  [ -9.3, -7.8]    0.0005    *
    edge  white   18.0     -7.3p  [ -7.9, -6.5]    0.0005    *

    edge  black    3.2     +1.4p  [ +1.0, +1.8]    0.0005    *
    edge  black    5.7     +1.6p  [ +1.1, +

## 9. Contrast 4 — position, with a difficulty control

Centre occlusion is harder even for a decoder that never saw occlusion, so A's own edge-to-centre
drop is the difficulty baseline. Only B's *excess* drop over A's is attributable to
generalisation.

In [ ]:
print(f"{'cells~':>7}{'A drop':>10}{'B drop':>10}{'B excess':>11}{'95% CI':>20}{'p':>10}")
print('-'*68)
for T in TARGETS:
    e, c = OUT[('edge','grey',T)], OUT[('center','grey',T)]
    dA = (e['A'].mean()-c['A'].mean())*100
    dB = (e['B'].mean()-c['B'].mean())*100
    # excess = (B_edge - B_ctr) - (A_edge - A_ctr), paired on markers
    diff_e = e['B'].astype(float) - e['A'].astype(float)
    diff_c = c['B'].astype(float) - c['A'].astype(float)
    d, lo, hi, p = paired(diff_e, diff_c, e['frame'])
    print(f"{e['cells']:>7.1f}{dA:>9.1f}p{dB:>9.1f}p{d*100:>10.1f}p  [{lo*100:>+5.1f},{hi*100:>+5.1f}]{p:>10.4f}")
print('\nB excess = (B_edge - B_centre) - (A_edge - A_centre).')
print('A positive excess is generalisation failure beyond intrinsic difficulty.')

json.dump({'levels': {f'{p}_{T}': LEVEL_FOR[(p,T)] for p in ('edge','center') for T in TARGETS},
           'achieved_cells': {f'{p}_{T}': float(ACHIEVED[(p,T)]) for p in ('edge','center') for T in TARGETS},
           'tests': [{'name':t[0],'cells':float(t[1]),'diff':float(t[2]),'lo':float(t[3]),'hi':float(t[4]),'p':float(t[5])} for t in TESTS],
           'n_tests': len(TESTS), 'n_significant_holm': int(sig.sum()),
           'reversal': {f'{k[0]}_{k[1]}': int(v) for k,v in rev.items()},
           'bootstrap': 'paired, cluster over frames, 2000 resamples',
           'correction': 'Holm-Bonferroni', 'viewpoints': DEV, 'heldout_untouched': HELDOUT},
          open(f'{ART}/n5_significance.json','w'), indent=1)
print(f'\nsaved -> {ART}/n5_significance.json')

 cells~    A drop    B drop   B excess              95% CI         p
--------------------------------------------------------------------
    3.2     12.0p     16.4p       4.4p  [ +3.5, +5.3]    0.0005
    5.7     23.5p     21.0p      -2.5p  [ -3.8, -1.2]    0.0005
    8.5     18.4p     25.2p       6.8p  [ +5.4, +8.3]    0.0005
   11.5      9.3p     26.4p      17.0p  [+15.5,+18.6]    0.0005
   15.6      2.6p     23.8p      21.2p  [+19.7,+22.6]    0.0005
   18.0      1.6p     24.7p      23.1p  [+21.9,+24.4]    0.0005

B excess = (B_edge - B_centre) - (A_edge - A_centre).
A positive excess is generalisation failure beyond intrinsic difficulty.

saved -> /content/drive/MyDrive/aruco_p2/artifacts/n5_significance.json
